In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
df = pd.read_csv("/kaggle/input/q1-ka-ai-2026/Q1_data.csv") # i got the path from the directory in the left panel of colab

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns



In [ ]:
# Task 5: Write your code here:
df["Delivery_Time"].hist(bins=30, edgecolor='black') # histogram for distribution

plt.title(f"Target Distribution ({'Delivery_Time'})")
plt.xlabel('Delivery_Time')
plt.ylabel("Frequency")
plt.grid(False)

plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop('Order_ID', axis=1)

In [ ]:
# Task 2: Write your code here:
df.isnull().sum()

In [ ]:
cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs']
df = df.dropna(subset= cols )


In [ ]:
# now for the Delivery time
df['Delivery_Time'] = df['Delivery_Time'].fillna(df['Delivery_Time'].mean())

In [ ]:
# Task 3: Write your code here:
duplicates = df.duplicated().sum()
print(f"Number of Duplicate Samples: {duplicates}")
if duplicates > 0:
  print("Dropping Duplicates...")
  df.drop_duplicates(inplace=True)

In [ ]:
df.head()

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder

categorical_cols = df.select_dtypes(exclude=["number"]) # get all object (strings) for encoding
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # drop target

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 6: Write your code here:
"""
its a regerssion task not a classification task (there is not classes to be imbalance)

"""

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error


mae_list = []

kf = KFold(n_splits=5, shuffle=True, random_state=42)   # kf not SKF since it is regression (so there is not imbalance in regression)


for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]


    model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)  # 100 random forrest model, with maximum depth of 20 for tree
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)


    mae_list.append(mean_absolute_error(y_test, y_pred))

print(f"MAE = {np.mean(mae_list):,.2f}")

In [ ]:

# Task 1: Write your code here:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False) # making highest importance features on top

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.hist(y_pred, bins=30)

plt.title(f" {'predicted: Delivery_Time'}")
plt.xlabel('Delivery_Time')
plt.ylabel("Frequency")
plt.grid(False)

plt.show()

In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost xgboost tqdm -q # for catboost

clear_output()

In [ ]:
# Task Bonus: Write your code here:
from catboost import CatBoostRegressor

all_mae_list = []

kf = KFold(n_splits=5, shuffle=True, random_state=42)


for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]


    model1 = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
    model2 = CatBoostRegressor(verbose=0)
    model1.fit(X_train, y_train) # random forest training
    model2.fit(X_train, y_train) # catboost training
    model1_pred = model1.predict(X_test) # random forest predict
    model2_pred = model2.predict(X_test) # catboost training predict


    all_mae_list.append(( mean_absolute_error(y_test, model1_pred) + mean_absolute_error(y_test, model2_pred)  )/ 2  ) # adding two mae then devide by 2 (mean)

print(f"MAE = {np.mean(mae_list):,.2f}")